# Matrix Decompositions - Examples

Practical demonstrations of LU, QR, and Cholesky decompositions.

## Contents
1. LU Decomposition
2. Solving Ax=b with LU
3. QR via Gram-Schmidt
4. QR Decomposition (NumPy)
5. Least Squares via QR
6. Cholesky Decomposition
7. Solving with Cholesky
8. Gaussian Sampling with Cholesky
9. Determinant from Decompositions
10. Condition Number and Stability
11. Comparing Methods

In [ ]:
import numpy as np
from numpy.linalg import qr, cholesky, inv, det, solve, lstsq
from scipy.linalg import lu, lu_factor, lu_solve
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

---
## 1. LU Decomposition

Factor $A = LU$ where $L$ is lower triangular and $U$ is upper triangular.

In [ ]:
A = np.array([[2, 1, 1],
              [4, 3, 3],
              [8, 7, 9]], dtype=float)

print(f"A = \n{A}")

In [ ]:
# Manual LU (without pivoting)
print("--- Manual LU (no pivoting) ---")
n = 3
L = np.eye(n)
U = A.copy()

for j in range(n - 1):
    for i in range(j + 1, n):
        L[i, j] = U[i, j] / U[j, j]
        U[i, :] = U[i, :] - L[i, j] * U[j, :]
        print(f"Eliminating A[{i},{j}]: multiplier = {L[i,j]:.4f}")

print(f"\nL = \n{np.round(L, 4)}")
print(f"\nU = \n{np.round(U, 4)}")
print(f"\nVerification: L @ U = \n{np.round(L @ U, 4)}")

In [ ]:
# SciPy LU with pivoting
print("--- SciPy LU (with pivoting) ---")
P, L_scipy, U_scipy = lu(A)
print(f"P (permutation) = \n{P}")
print(f"L = \n{np.round(L_scipy, 4)}")
print(f"U = \n{np.round(U_scipy, 4)}")
print(f"\nP @ L @ U = \n{np.round(P @ L_scipy @ U_scipy, 4)}")

---
## 2. Solving Ax = b with LU

Factor once, then solve efficiently for multiple right-hand sides.

In [ ]:
A = np.array([[2, 1],
              [4, 5]], dtype=float)
b = np.array([3, 17], dtype=float)

print(f"A = \n{A}")
print(f"b = {b}")

# Factor once
lu_factored, piv = lu_factor(A)
print(f"\nLU factored (stored in one matrix):\n{np.round(lu_factored, 4)}")

# Solve
x = lu_solve((lu_factored, piv), b)
print(f"\nSolution x = {x}")
print(f"Verification: A @ x = {A @ x}")

In [ ]:
# Multiple right-hand sides
print("--- Multiple right-hand sides ---")
B = np.array([[3, 1, 2],
              [17, 5, 8]], dtype=float)

X = lu_solve((lu_factored, piv), B)
print(f"Solutions for 3 different b vectors:\n{X}")
print(f"\nVerification: A @ X = \n{A @ X}")

---
## 3. QR via Gram-Schmidt

Orthonormalize columns of $A$ to get $Q$, compute projection coefficients for $R$.

In [ ]:
A = np.array([[1, 1, 0],
              [1, 0, 1],
              [0, 1, 1]], dtype=float)

print(f"A = \n{A}")

# Manual Gram-Schmidt
print("\n--- Gram-Schmidt Process ---")
n = A.shape[1]
Q = np.zeros_like(A)
R = np.zeros((n, n))

for j in range(n):
    v = A[:, j].copy()
    print(f"\nProcessing column {j+1}: a_{j+1} = {A[:, j]}")
    
    # Subtract projections
    for i in range(j):
        R[i, j] = Q[:, i] @ A[:, j]
        v = v - R[i, j] * Q[:, i]
        print(f"  Projection onto q_{i+1}: {R[i,j]:.4f}")
    
    R[j, j] = np.linalg.norm(v)
    Q[:, j] = v / R[j, j]
    print(f"  Norm: {R[j,j]:.4f}")
    print(f"  q_{j+1} = {np.round(Q[:, j], 4)}")

print(f"\nQ = \n{np.round(Q, 4)}")
print(f"\nR = \n{np.round(R, 4)}")
print(f"\nQᵀQ = \n{np.round(Q.T @ Q, 4)} (should be I)")
print(f"\nQ @ R = \n{np.round(Q @ R, 4)}")

---
## 4. QR Decomposition (NumPy)

Comparing reduced and full QR decomposition.

In [ ]:
A = np.array([[1, 2],
              [3, 4],
              [5, 6]], dtype=float)

print(f"A (3×2) = \n{A}")

# Reduced QR
Q, R = qr(A, mode='reduced')
print(f"\n--- Reduced QR ---")
print(f"Q (3×2) = \n{np.round(Q, 4)}")
print(f"R (2×2) = \n{np.round(R, 4)}")
print(f"QᵀQ = \n{np.round(Q.T @ Q, 4)}")

# Full QR
Q_full, R_full = qr(A, mode='complete')
print(f"\n--- Full QR ---")
print(f"Q (3×3) = \n{np.round(Q_full, 4)}")
print(f"R (3×2) = \n{np.round(R_full, 4)}")

---
## 5. Least Squares via QR

For overdetermined systems, QR gives the least-squares solution.

In [ ]:
# Fitting line y = c + mx to 4 points
X = np.array([[1, 0],
              [1, 1],
              [1, 2],
              [1, 3]], dtype=float)  # [1, x] columns
y = np.array([1, 2.5, 3.5, 5], dtype=float)

print("Fitting line y = c + mx to points:")
print("x: [0, 1, 2, 3]")
print(f"y: {y}")

print(f"\nDesign matrix X = \n{X}")

# QR approach
Q, R = qr(X, mode='reduced')
print(f"\nQ = \n{np.round(Q, 4)}")
print(f"\nR = \n{np.round(R, 4)}")

# Solve R @ coef = Q^T @ y
coef = solve(R, Q.T @ y)
print(f"\nCoefficients [c, m] = {np.round(coef, 4)}")
print(f"Best fit line: y = {coef[0]:.4f} + {coef[1]:.4f}x")

In [ ]:
# Visualize the fit
x_points = np.array([0, 1, 2, 3])
x_line = np.linspace(-0.5, 3.5, 100)
y_line = coef[0] + coef[1] * x_line

plt.figure(figsize=(8, 5))
plt.scatter(x_points, y, s=100, c='red', zorder=5, label='Data points')
plt.plot(x_line, y_line, 'b-', linewidth=2, label=f'Fit: y = {coef[0]:.2f} + {coef[1]:.2f}x')

# Show residuals
y_pred = X @ coef
for i in range(4):
    plt.plot([x_points[i], x_points[i]], [y[i], y_pred[i]], 'g--', alpha=0.5)

plt.xlabel('x'); plt.ylabel('y')
plt.title('Least Squares Fit via QR Decomposition')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Residual norm: {np.linalg.norm(X @ coef - y):.4f}")

---
## 6. Cholesky Decomposition

For symmetric positive definite $A = LL^T$.

In [ ]:
A = np.array([[4, 2, 2],
              [2, 10, 7],
              [2, 7, 21]], dtype=float)

print(f"A (symmetric positive definite) = \n{A}")

# Manual Cholesky
print("\n--- Manual Cholesky ---")
n = 3
L = np.zeros((n, n))

for j in range(n):
    # Diagonal
    sum_sq = sum(L[j, k]**2 for k in range(j))
    L[j, j] = np.sqrt(A[j, j] - sum_sq)
    print(f"L[{j},{j}] = √({A[j,j]} - {sum_sq:.4f}) = {L[j,j]:.4f}")
    
    # Below diagonal
    for i in range(j + 1, n):
        sum_prod = sum(L[i, k] * L[j, k] for k in range(j))
        L[i, j] = (A[i, j] - sum_prod) / L[j, j]

print(f"\nL = \n{np.round(L, 4)}")
print(f"\nL @ Lᵀ = \n{np.round(L @ L.T, 4)}")

In [ ]:
# NumPy Cholesky
print("--- NumPy Cholesky ---")
L_np = cholesky(A)
print(f"L = \n{np.round(L_np, 4)}")

---
## 7. Solving with Cholesky

Forward and back substitution for $Ax = b$ using $A = LL^T$.

In [ ]:
A = np.array([[4, 2],
              [2, 5]], dtype=float)
b = np.array([8, 11], dtype=float)

print(f"A = \n{A}")
print(f"b = {b}")

# Cholesky factor
L = cholesky(A)
print(f"\nL = \n{L}")

# Forward substitution: Ly = b
y = np.zeros(2)
y[0] = b[0] / L[0, 0]
y[1] = (b[1] - L[1, 0] * y[0]) / L[1, 1]
print(f"\nForward solve Ly = b: y = {y}")

# Back substitution: L^T x = y
LT = L.T
x = np.zeros(2)
x[1] = y[1] / LT[1, 1]
x[0] = (y[0] - LT[0, 1] * x[1]) / LT[0, 0]
print(f"Back solve Lᵀx = y: x = {x}")

print(f"\nVerification: A @ x = {A @ x}")

---
## 8. Gaussian Sampling with Cholesky

Sample from $\mathcal{N}(\mu, \Sigma)$ using $x = \mu + L \cdot z$ where $z \sim \mathcal{N}(0, I)$.

In [ ]:
# Define Gaussian N(mu, Sigma)
mu = np.array([1, 2])
Sigma = np.array([[2, 1],
                  [1, 3]])

print(f"Mean: μ = {mu}")
print(f"Covariance: Σ = \n{Sigma}")

# Cholesky factor
L = cholesky(Sigma)
print(f"\nCholesky L = \n{np.round(L, 4)}")

# Sample
np.random.seed(42)
n_samples = 500

print("\n--- Sampling Process ---")
print("For each sample: x = μ + L @ z, where z ~ N(0, I)")

samples = []
for i in range(n_samples):
    z = np.random.randn(2)
    x = mu + L @ z
    samples.append(x)

samples = np.array(samples)
print(f"\nSample mean: {np.round(samples.mean(axis=0), 3)}")
print(f"True mean: {mu}")
print(f"\nSample covariance:\n{np.round(np.cov(samples.T), 3)}")
print(f"True covariance:\n{Sigma}")

In [ ]:
# Visualize samples
plt.figure(figsize=(8, 6))
plt.scatter(samples[:, 0], samples[:, 1], alpha=0.3, s=20, c='blue')
plt.scatter([mu[0]], [mu[1]], c='red', s=200, marker='*', label='Mean μ', zorder=5)

# Draw covariance ellipse
eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
angle = np.degrees(np.arctan2(eigenvectors[1, 1], eigenvectors[0, 1]))
from matplotlib.patches import Ellipse
for scale in [1, 2]:
    ellipse = Ellipse(xy=mu, width=2*scale*np.sqrt(eigenvalues[1]), 
                      height=2*scale*np.sqrt(eigenvalues[0]), angle=angle,
                      fill=False, color='red', linewidth=2)
    plt.gca().add_patch(ellipse)

plt.xlabel('$x_1$'); plt.ylabel('$x_2$')
plt.title('Gaussian Samples via Cholesky Decomposition')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()

---
## 9. Determinant from Decompositions

Computing $\det(A)$ efficiently using decomposition.

In [ ]:
A = np.array([[3, 1, 2],
              [1, 4, 1],
              [2, 1, 5]], dtype=float)

print(f"A = \n{A}")
print(f"\nDirect: det(A) = {det(A):.4f}")

# From LU
P, L, U = lu(A)
det_lu = np.prod(np.diag(U)) * det(P)
print(f"\nFrom LU: det = det(P) × ∏ u_ii = {det_lu:.4f}")

# From Cholesky (A is SPD)
try:
    L_chol = cholesky(A)
    det_chol = np.prod(np.diag(L_chol))**2
    print(f"From Cholesky: det = (∏ l_ii)² = {det_chol:.4f}")
except:
    print("Cholesky failed (A not PD)")

---
## 10. Condition Number and Stability

Ill-conditioned systems amplify numerical errors.

In [ ]:
# Well-conditioned system
A_good = np.array([[1, 0],
                   [0, 1]], dtype=float)

# Ill-conditioned system
A_bad = np.array([[1, 1],
                  [1, 1.0001]], dtype=float)

for name, A in [("Well-conditioned (I)", A_good), ("Ill-conditioned", A_bad)]:
    print(f"\n{name} system:")
    print(f"A = \n{A}")
    
    cond = np.linalg.cond(A)
    print(f"Condition number κ(A) = {cond:.2f}")
    
    b = np.array([2, 2], dtype=float)
    x = solve(A, b)
    print(f"Solution for b = {b}: x = {np.round(x, 4)}")
    
    # Perturb b slightly
    b_perturbed = b + np.array([0.0001, 0])
    x_perturbed = solve(A, b_perturbed)
    print(f"Solution for b + [0.0001, 0]: x = {np.round(x_perturbed, 4)}")
    print(f"Change in x: {np.linalg.norm(x_perturbed - x):.4f}")

---
## 11. Comparing Methods for Ax = b

Benchmarking different decomposition approaches.

In [ ]:
import time

np.random.seed(42)
n = 200

# Create SPD matrix
B = np.random.randn(n, n)
A = B.T @ B + 0.1 * np.eye(n)  # Ensure PD
b = np.random.randn(n)

print(f"System size: {n} × {n}")

methods = []

# Direct solve
t0 = time.time()
x_direct = solve(A, b)
t_direct = time.time() - t0
methods.append(("Direct", t_direct*1000, np.linalg.norm(A @ x_direct - b)))

# LU
t0 = time.time()
lu_piv = lu_factor(A)
x_lu = lu_solve(lu_piv, b)
t_lu = time.time() - t0
methods.append(("LU", t_lu*1000, np.linalg.norm(A @ x_lu - b)))

# Cholesky
t0 = time.time()
L = cholesky(A)
y = solve(L, b)
x_chol = solve(L.T, y)
t_chol = time.time() - t0
methods.append(("Cholesky", t_chol*1000, np.linalg.norm(A @ x_chol - b)))

# QR
t0 = time.time()
Q, R = qr(A)
x_qr = solve(R, Q.T @ b)
t_qr = time.time() - t0
methods.append(("QR", t_qr*1000, np.linalg.norm(A @ x_qr - b)))

print("\nMethod      | Time (ms) | Residual norm")
print("-" * 45)
for name, t, res in methods:
    print(f"{name:11s} | {t:8.3f}  | {res:.2e}")

print("\nCholesky is typically fastest for SPD matrices!")

In [ ]:
# Visualize timing comparison
names = [m[0] for m in methods]
times = [m[1] for m in methods]

plt.figure(figsize=(8, 5))
bars = plt.bar(names, times, color=['steelblue', 'orange', 'green', 'purple'])
plt.ylabel('Time (ms)')
plt.title(f'Decomposition Methods Timing ({n}×{n} SPD matrix)')

# Add value labels
for bar, t in zip(bars, times):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{t:.2f}', ha='center', va='bottom')

plt.grid(axis='y', alpha=0.3)
plt.show()

---
## Summary

### Decomposition Overview

| Decomposition | Formula | Requirements | Main Use |
|---------------|---------|--------------|----------|
| LU | $A = LU$ or $PA = LU$ | Square | Linear systems |
| QR | $A = QR$ | Any (m ≥ n) | Least squares, eigenvalues |
| Cholesky | $A = LL^T$ | Symmetric PD | Fast systems, sampling |

### When to Use Each Method

```
Solving Ax = b:
│
├── A is symmetric positive definite?
│   └── YES → Cholesky (fastest, most stable)
│
├── A is square?
│   └── YES → LU with pivoting
│
├── A is rectangular (overdetermined)?
│   └── YES → QR for least squares
│
└── Need most general solution?
    └── SVD (next section)
```

### ML Applications

- **Ridge Regression**: Cholesky on $(X^TX + \lambda I)$
- **Gaussian Sampling**: Cholesky of covariance
- **Linear Regression**: QR on design matrix
- **Solving Normal Equations**: LU or Cholesky